# 06. Machine Learning Models

In [24]:
# Import libraries
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

# Add project root to path
import os
import sys
sys.path.append(os.path.abspath('../'))
from src.data_loader import load_data, save_file
from src.splitting import split_train_test
from src.evaluation import evaluate, highlight_table
from src.models.baseline import naive_forecast, seasonal_naive_forecast
from src.models.ml import *

# Visualization settings
plt.style.use('seaborn-v0_8-darkgrid')
sns.set_palette("husl")
pd.set_option("display.max_columns", 50)
pd.set_option("display.width", 120)

print(f"Python version: {sys.version.split()[0]}")
print(f"Pandas version: {pd.__version__}")
print(f"NumPy version: {np.__version__}")

Python version: 3.10.11
Pandas version: 2.1.4
NumPy version: 1.26.4


In [25]:
# Load data using custom loader
try:
    df = load_data('data/processed/data_total_features.parquet')
    print("Data loaded successfully!")
    print(f"Shape: {df.shape}")
    print(f"Time span: {df.index.min()} → {df.index.max()}")
    if not df.index.freq:
        df = df.asfreq("1H")
    print(f"Frequency: {df.index.freq}")
except Exception as e:
    print(f"[Error] {e}")

Loading: data/processed/data_total_features.parquet
Data loaded successfully!
Shape: (35064, 11)
Time span: 2011-01-01 00:00:00 → 2014-12-31 23:00:00
Frequency: <Hour>


In [26]:
df_split = df.copy()
df_split = df_split[df.index >= "2012-01-01"]
df_split.head()

,total_load,hour,month,day_of_week,day_of_month,day_of_year,hour_sin,hour_cos,year_sin,year_cos,holiday_name
Timestamp,,,,,,,,,,,
2012-01-01 00:00:00,104230.493821,0,1,6,1,1,0.000000,1.000000,0.0,1.0,Ano Novo
2012-01-01 01:00:00,112550.368868,1,1,6,1,1,0.258819,0.965926,0.0,1.0,Ano Novo
2012-01-01 02:00:00,110930.247627,2,1,6,1,1,0.500000,0.866025,0.0,1.0,Ano Novo
2012-01-01 03:00:00,106191.754971,3,1,6,1,1,0.707107,0.707107,0.0,1.0,Ano Novo
2012-01-01 04:00:00,103540.419166,4,1,6,1,1,0.866025,0.500000,0.0,1.0,Ano Novo


In [27]:
X_train, X_test, y_train, y_test, _ = split_train_test(df_split, 'total_load', split_time="2014-01-01", categorical_cols='holiday_name')

print("Train shape:", X_train.shape)
print("Test shape:", X_test.shape)

Train shape: (17568, 10)
Test shape: (8759, 10)


## 1. Baseline

In [28]:
# Naive forecast
seasonal_naive_pred = seasonal_naive_forecast(df_split, y_test, season=365)

# Evaluate forecasts
seasonal_naive_metrics = evaluate(y_test, seasonal_naive_pred)

print("Seasonal Naive (24h):", seasonal_naive_metrics)

Seasonal Naive (24h): {'MAE': 14598.19, 'RMSE': 21626.9, 'MAPE': 6.07}


## 2. Models

### 2.1 Random Forest

In [ ]:
rdf, rdf_pred = random_forest_model(X_train, y_train, X_test)
rdf_metrics = evaluate(y_test, rdf_pred)

print("Random Forest:", rdf_metrics)

LightGBM: {'MAE': 13265.68, 'RMSE': 19425.27, 'MAPE': 5.56}


### 2.2 LightGBM

In [30]:
lgbm, lgbm_pred = lightgbm_model(X_train, y_train, X_test)
lgbm_metrics = evaluate(y_test, lgbm_pred)

print("LightGBM:", lgbm_metrics)

LightGBM: {'MAE': 13629.68, 'RMSE': 19854.71, 'MAPE': 5.8}


### 2.3 XGBoost

In [31]:
xgb, xgb_pred = xgboost_model(X_train, y_train, X_test)
xgb_metrics = evaluate(y_test, xgb_pred)

print("XGBoost:", xgb_metrics)

XGBoost: {'MAE': 13570.01, 'RMSE': 20275.23, 'MAPE': 5.63}


## 3. Revive 2011

In [32]:
import requests
import numpy as np
import pandas as pd
from lightgbm import LGBMRegressor
from sklearn.compose import TransformedTargetRegressor

# Đảm bảo df_clean gốc được sắp xếp chuẩn theo DatetimeIndex
df_clean = df.sort_index()
target = 'total_load'

print("--- BẮT ĐẦU QUY TRÌNH: TẢI API THỜI TIẾT & PHỤC DỰNG 2011 ---")

# =========================================================================
# BƯỚC 1: TẢI DỮ LIỆU THỜI TIẾT TỪ OPEN-METEO API (MIỄN PHÍ)
# =========================================================================
# Sử dụng tọa độ Lisbon, Bồ Đào Nha (Lat: 38.7167, Lon: -9.1333) làm proxy
# Chỉ cần tải từ 2011-01-01 đến 2013-12-31 (phục vụ riêng cho Backcasting)
print("1. Đang kết nối API Open-Meteo tải dữ liệu thời tiết quá khứ...")

url = (
    "https://archive-api.open-meteo.com/v1/archive?"
    "latitude=38.7167&longitude=-9.1333"
    "&start_date=2011-01-01&end_date=2013-12-31"
    "&hourly=temperature_2m,relative_humidity_2m,wind_speed_10m"
    "&timezone=Europe%2FLisbon"
)

response = requests.get(url)
if response.status_code != 200:
    raise Exception(f"Lỗi tải API: {response.text}")

data = response.json()
hourly = data['hourly']

# Chuyển đổi JSON thành DataFrame
df_weather_api = pd.DataFrame({
    'time': pd.to_datetime(hourly['time']),
    'temp': hourly['temperature_2m'],
    'humidity': hourly['relative_humidity_2m'],
    'wind_speed': hourly['wind_speed_10m']
}).set_index('time')

# Loại bỏ thông tin múi giờ (tz-naive) để an toàn gộp với df_clean của bạn
if df_weather_api.index.tz is not None:
    df_weather_api.index = df_weather_api.index.tz_localize(None)

print(f"-> Tải thành công {len(df_weather_api)} dòng dữ liệu thời tiết!")


# =========================================================================
# BƯỚC 2: GỘP DỮ LIỆU & FEATURE ENGINEERING THỜI TIẾT
# =========================================================================
print("2. Gộp dữ liệu và trích xuất các đặc trưng khí hậu bậc cao...")

# Gộp (Join) thời tiết vào bản sao của df_clean dựa trên khớp mốc giờ
df_process = df_clean.loc['2011-01-01':'2013-12-31'].copy()
df_process = df_process.join(df_weather_api, how='left')

# Xử lý nội suy nếu API có sót vài giờ ngẫu nhiên
df_process[['temp', 'humidity', 'wind_speed']] = df_process[['temp', 'humidity', 'wind_speed']].interpolate(method='time')

# --- Tạo Features Khí hậu ---
# 2.1. CDD/HDD (Làm mát > 18°C, Sưởi ấm < 15°C)
df_process['CDD'] = np.maximum(0, df_process['temp'] - 18)
df_process['HDD'] = np.maximum(0, 15 - df_process['temp'])
df_process['CDD_squared'] = df_process['CDD'] ** 2

# 2.2. Quán tính nhiệt (Ngậm nhiệt sau 3 ngày nắng gắt)
df_process['temp_rolling_mean_72h'] = df_process['temp'].rolling(window=72).mean()

# 2.3. Chỉ số oi bức (Apparent Temperature / THI)
T = df_process['temp']
RH = df_process['humidity']
df_process['THI_Apparent'] = T - (0.55 - 0.0055 * RH) * (T - 14.5)

# 2.4. Gió làm mát khi trời đang nóng
df_process['wind_cooling_effect'] = df_process['wind_speed'] * df_process['CDD']

# 2.5. Tương tác hành vi: Giờ x Nhiệt độ
if 'hour' in df_process.columns:
    df_process['hour_x_CDD'] = df_process['hour'] * df_process['CDD']

# Lấp đầy mốc NaN do rolling ở các dòng đầu
df_process = df_process.bfill()


# =========================================================================
# BƯỚC 3: HUẤN LUYỆN "CỖ MÁY THỜI GIAN" VỚI LIGHTGBM
# =========================================================================
# Lấy toàn bộ features ngoại sinh gốc + features thời tiết vừa tạo
exo_features = [col for col in df_process.columns if col != target]

# Tập Train: Giai đoạn trưởng thành (2012 và 2013)
X_train_stable = df_process.loc['2012-01-01':'2013-12-31', exo_features]
y_train_stable = df_process.loc['2012-01-01':'2013-12-31', target]

# Tập Cần phục dựng (Hindcast): Toàn bộ năm 2011
X_backcast_2011 = df_process.loc['2011-01-01':'2011-12-31', exo_features]

lgbm_core = LGBMRegressor(
    n_estimators=1000,
    learning_rate=0.03,
    num_leaves=63,
    colsample_bytree=0.7,
    random_state=42,
    n_jobs=-1
)

backcast_model = TransformedTargetRegressor(
    regressor=lgbm_core,
    func=np.log1p,
    inverse_func=np.expm1
)

print("3. Đang huấn luyện mô hình học quy luật tải theo thời tiết từ 2012-2013...")
backcast_model.fit(X_train_stable, y_train_stable)


# =========================================================================
# BƯỚC 4: TÁI TẠO 2011 & TRẢ VỀ KẾT QUẢ SẠCH (BỎ THỜI TIẾT)
# =========================================================================
print("4. Đang ánh xạ ngược để tái tạo sản lượng điện năm 2011...")
y_synthetic_2011 = backcast_model.predict(X_backcast_2011)

# Lấy lại tập df_clean GỐC của bạn (để đảm bảo không dính dáng gì đến cột thời tiết)
df_2011 = df.copy()

# THAY THẾ HOÀN TOÀN tải điện thô 2011 bằng chuỗi tổng hợp chuẩn thời tiết
df_2011.loc['2011-01-01':'2011-12-31', target] = y_synthetic_2011

print("--- HOÀN TẤT QUY TRÌNH! ---")
print("-> Toàn bộ dữ liệu thời tiết tải từ API đã làm xong nhiệm vụ đòn bẩy và ĐÃ BỊ LOẠI BỎ.")
print("-> Tập df_2011 hiện tại giữ nguyên cấu trúc cột ban đầu của bạn.")
print("-> Đỉnh tải điện 2011 đã được nâng khớp hoàn hảo. Bạn có thể tạo lag_365 ngay!")

--- BẮT ĐẦU QUY TRÌNH: TẢI API THỜI TIẾT & PHỤC DỰNG 2011 ---
1. Đang kết nối API Open-Meteo tải dữ liệu thời tiết quá khứ...
-> Tải thành công 26304 dòng dữ liệu thời tiết!
2. Gộp dữ liệu và trích xuất các đặc trưng khí hậu bậc cao...
3. Đang huấn luyện mô hình học quy luật tải theo thời tiết từ 2012-2013...
4. Đang ánh xạ ngược để tái tạo sản lượng điện năm 2011...
--- HOÀN TẤT QUY TRÌNH! ---
-> Toàn bộ dữ liệu thời tiết tải từ API đã làm xong nhiệm vụ đòn bẩy và ĐÃ BỊ LOẠI BỎ.
-> Tập df_2011 hiện tại giữ nguyên cấu trúc cột ban đầu của bạn.
-> Đỉnh tải điện 2011 đã được nâng khớp hoàn hảo. Bạn có thể tạo lag_365 ngay!


In [33]:
df_2011.head()

,total_load,hour,month,day_of_week,day_of_month,day_of_year,hour_sin,hour_cos,year_sin,year_cos,holiday_name
Timestamp,,,,,,,,,,,
2011-01-01 00:00:00,128273.479673,0,1,5,1,1,0.000000,1.000000,0.0,1.0,Ano Novo
2011-01-01 01:00:00,109173.297911,1,1,5,1,1,0.258819,0.965926,0.0,1.0,Ano Novo
2011-01-01 02:00:00,105270.467688,2,1,5,1,1,0.500000,0.866025,0.0,1.0,Ano Novo
2011-01-01 03:00:00,103119.465761,3,1,5,1,1,0.707107,0.707107,0.0,1.0,Ano Novo
2011-01-01 04:00:00,100037.338338,4,1,5,1,1,0.866025,0.500000,0.0,1.0,Ano Novo


In [34]:
X_train, X_test, y_train, y_test, _ = split_train_test(df_2011, 'total_load', split_time="2014-01-01", categorical_cols='holiday_name')

print("Train shape:", X_train.shape)
print("Test shape:", X_test.shape)

Train shape: (26328, 10)
Test shape: (8759, 10)


In [35]:
rdf, rdf_pred = random_forest_model(X_train, y_train, X_test)
rdf_metrics = evaluate(y_test, rdf_pred)

print("Random Forest:", rdf_metrics)

Random Forest: {'MAE': 12828.45, 'RMSE': 18776.22, 'MAPE': 5.36}


In [36]:
lgbm, lgbm_pred = lightgbm_model(X_train, y_train, X_test)
lgbm_metrics = evaluate(y_test, lgbm_pred)

print("LightGBM:", lgbm_metrics)

LightGBM: {'MAE': 13012.48, 'RMSE': 18869.57, 'MAPE': 5.58}


In [37]:
xgb, xgb_pred = xgboost_model(X_train, y_train, X_test)
xgb_metrics = evaluate(y_test, xgb_pred)

print("XGBoost:", xgb_metrics)

XGBoost: {'MAE': 12965.75, 'RMSE': 19446.47, 'MAPE': 5.37}


## 4. Optimization

In [56]:
from src.models.optimizations import *

c:\Users\YOGA\Desktop\01_Time Series\PROJECT\Github\main\Electricity-Load-Diagrams\.venv\lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [61]:
df_optimize = df_2011.copy()
df_optimize = df_optimize[df.index < '2014-01-01']
df_optimize.head()

,total_load,hour,month,day_of_week,day_of_month,day_of_year,hour_sin,hour_cos,year_sin,year_cos,holiday_name
Timestamp,,,,,,,,,,,
2011-01-01 00:00:00,128273.479673,0,1,5,1,1,0.000000,1.000000,0.0,1.0,Ano Novo
2011-01-01 01:00:00,109173.297911,1,1,5,1,1,0.258819,0.965926,0.0,1.0,Ano Novo
2011-01-01 02:00:00,105270.467688,2,1,5,1,1,0.500000,0.866025,0.0,1.0,Ano Novo
2011-01-01 03:00:00,103119.465761,3,1,5,1,1,0.707107,0.707107,0.0,1.0,Ano Novo
2011-01-01 04:00:00,100037.338338,4,1,5,1,1,0.866025,0.500000,0.0,1.0,Ano Novo


In [ ]:
X_train, X_valid, y_train, y_valid, _ = split_train_test(df_optimize, 'total_load', split_time="2013-01-01", categorical_cols='holiday_name')

print("Train shape:", X_train.shape)
print("Val shape:", X_valid.shape)

Train shape: (17568, 10)
Test shape: (8759, 10)


In [72]:
rdf_best = optuna_random_forest(X_train, y_train, X_valid, y_valid, n_trials=50)

[I 2026-05-12 20:54:12,691] A new study created in memory with name: no-name-b80955f3-9192-4c26-bc0e-4c66aa99b5a5
[I 2026-05-12 20:54:14,665] Trial 0 finished with value: 20794.616670641557 and parameters: {'n_estimators': 437, 'max_depth': 48, 'min_samples_split': 15, 'min_samples_leaf': 6, 'max_features': 0.5780093202212182}. Best is trial 0 with value: 20794.616670641557.
[I 2026-05-12 20:54:15,899] Trial 1 finished with value: 20549.199834501498 and parameters: {'n_estimators': 240, 'max_depth': 12, 'min_samples_split': 18, 'min_samples_leaf': 7, 'max_features': 0.8540362888980227}. Best is trial 1 with value: 20549.199834501498.
[I 2026-05-12 20:54:16,459] Trial 2 finished with value: 20765.13226391077 and parameters: {'n_estimators': 118, 'max_depth': 49, 'min_samples_split': 17, 'min_samples_leaf': 3, 'max_features': 0.5909124836035503}. Best is trial 1 with value: 20549.199834501498.
[I 2026-05-12 20:54:17,783] Trial 3 finished with value: 20883.855046794895 and parameters: {'n

MODEL      : random_forest
BEST RMSE  : 20283.7575
BEST PARAMS:
{'n_estimators': 914, 'max_depth': 10, 'min_samples_split': 17, 'min_samples_leaf': 2, 'max_features': 0.8368078049885144}


In [63]:
lgbm_best = optuna_lightgbm(X_train, y_train, X_valid, y_valid, n_trials=50)

[I 2026-05-12 20:45:57,045] A new study created in memory with name: no-name-6a8f0e84-4c3e-47b3-8569-2b1751870ee1
[I 2026-05-12 20:46:02,627] Trial 0 finished with value: 21549.73951115825 and parameters: {'n_estimators': 1311, 'learning_rate': 0.044635901521768134, 'max_depth': 10, 'num_leaves': 159, 'subsample': 0.6624074561769746, 'colsample_bytree': 0.662397808134481, 'min_child_samples': 10, 'reg_alpha': 8.661761457749352, 'reg_lambda': 6.011150117432088}. Best is trial 0 with value: 21549.73951115825.
[I 2026-05-12 20:46:13,977] Trial 1 finished with value: 21264.912357945133 and parameters: {'n_estimators': 2212, 'learning_rate': 0.005242693862597309, 'max_depth': 12, 'num_leaves': 215, 'subsample': 0.6849356442713105, 'colsample_bytree': 0.6727299868828402, 'min_child_samples': 22, 'reg_alpha': 3.0424224295953772, 'reg_lambda': 5.247564316322379}. Best is trial 1 with value: 21264.912357945133.
[I 2026-05-12 20:46:16,517] Trial 2 finished with value: 21137.49535045469 and param

MODEL      : lightgbm
BEST RMSE  : 20020.4306
BEST PARAMS:
{'n_estimators': 937, 'learning_rate': 0.007791311812192606, 'max_depth': 4, 'num_leaves': 176, 'subsample': 0.9954627320949059, 'colsample_bytree': 0.9993134344664012, 'min_child_samples': 30, 'reg_alpha': 0.002015220146534741, 'reg_lambda': 7.285146928910762}


In [71]:
xgb_best = optuna_xgboost(X_train, y_train, X_valid, y_valid, n_trials=50)

[I 2026-05-12 20:52:46,038] A new study created in memory with name: no-name-ad0e19b9-050b-4362-ac5a-790c3ac08205
[I 2026-05-12 20:52:50,481] Trial 0 finished with value: 21801.522498618615 and parameters: {'n_estimators': 1311, 'learning_rate': 0.044635901521768134, 'max_depth': 10, 'subsample': 0.8394633936788146, 'colsample_bytree': 0.6624074561769746, 'min_child_weight': 4, 'gamma': 0.5808361216819946, 'reg_alpha': 8.661761457749352, 'reg_lambda': 6.011150117432088}. Best is trial 0 with value: 21801.522498618615.
[I 2026-05-12 20:53:01,662] Trial 1 finished with value: 21190.239331705667 and parameters: {'n_estimators': 2212, 'learning_rate': 0.005242693862597309, 'max_depth': 12, 'subsample': 0.9329770563201687, 'colsample_bytree': 0.6849356442713105, 'min_child_weight': 4, 'gamma': 1.8340450985343382, 'reg_alpha': 3.0424224295953772, 'reg_lambda': 5.247564316322379}. Best is trial 1 with value: 21190.239331705667.
[I 2026-05-12 20:53:05,447] Trial 2 finished with value: 21254.02

MODEL      : xgboost
BEST RMSE  : 19990.3220
BEST PARAMS:
{'n_estimators': 578, 'learning_rate': 0.011690156280708838, 'max_depth': 4, 'subsample': 0.8877027290749933, 'colsample_bytree': 0.974481026311757, 'min_child_weight': 18, 'gamma': 1.9207645291808602, 'reg_alpha': 8.537671637807994, 'reg_lambda': 3.92228062985356}


In [73]:
X_train, X_test, y_train, y_test, _ = split_train_test(df_2011, 'total_load', split_time="2014-01-01", categorical_cols='holiday_name')

print("Train shape:", X_train.shape)
print("Test shape:", X_test.shape)

Train shape: (26328, 10)
Test shape: (8759, 10)


In [74]:
rdf, rdf_pred = random_forest_model(X_train, y_train, X_test, rdf_best.best_params)
rdf_metrics = evaluate(y_test, rdf_pred)

print("Random Forest:", rdf_metrics)

Random Forest: {'MAE': 13093.43, 'RMSE': 18439.95, 'MAPE': 5.75}


In [76]:
lgbm, lgbm_pred = lightgbm_model(X_train, y_train, X_test, lgbm_best.best_params)
lgbm_metrics = evaluate(y_test, lgbm_pred)

print("LightGBM:", lgbm_metrics)

LightGBM: {'MAE': 13063.47, 'RMSE': 18310.15, 'MAPE': 5.71}


In [77]:
xgb, xgb_pred = xgboost_model(X_train, y_train, X_test, xgb_best.best_params)
xgb_metrics = evaluate(y_test, xgb_pred)

print("XGBoost:", xgb_metrics)

XGBoost: {'MAE': 14012.4, 'RMSE': 19834.35, 'MAPE': 6.01}


## 5. Tổng hợp và So sánh Kết quả (Compare)

Dưới đây là bảng tổng hợp hiệu suất của các mô hình qua 3 giai đoạn: Trước khi có dữ liệu 2011, Sau khi phục dựng 2011 (thêm features), và Sau khi tinh chỉnh siêu tham số (Hyperparameter Tuning).

| Giai đoạn | Mô hình | MAE | RMSE | MAPE (%) |
| :--- | :--- | :--- | :--- | :--- |
| **Baseline** | Seasonal Naive (24h) | 14,598.19 | 21,626.90 | 6.07 |
| | | | | |
| **Thiếu năm 2011** | Random Forest / Base 1 | 13,265.68 | 19,425.27 | 5.56 |
| | LightGBM | 13,629.68 | 19,854.71 | 5.80 |
| | XGBoost | 13,570.01 | 20,275.23 | 5.63 |
| | | | | |
| **Có năm 2011 (Backcast)** | Random Forest | **12,828.45** | 18,776.22 | **5.36** |
|  | LightGBM | 13,012.48 | 18,869.57 | 5.58 |
| | XGBoost | 12,965.75 | 19,446.47 | 5.37 |
| | | | | |
| **Sau Tối ưu hóa (Optuna)** | Random Forest (Tuned) | 13,093.43 | 18,439.95 | 5.75 |
| *(Mục tiêu: Min RMSE)* | LightGBM (Tuned) | 13,063.47 | **18,310.15** | 5.71 |
| | XGBoost (Tuned) | 14,012.40 | 19,834.35 | 6.01 |
